# dutil

> dutil provides dialog introspection, message linking, tool discovery, and import/export utilities for solveit dialogs.

In [ ]:
#| default_exp dutil

In [ ]:
#|hide
gk = tuple((g := globals()).keys())

In [ ]:
import asyncio, os, re, sys, inspect, uuid
from collections import defaultdict
from inspect import Parameter, currentframe
from pathlib import Path
from typing import Any, Mapping
from IPython import get_ipython
import tiktoken
from fastcore.aio import maybe_await
from fastcore.basics import groupby
from fastcore.imports import in_ipython
from fastcore.meta import delegates
from fastcore.xtras import is_listy, fenced_blocks
from fasthtml.components import Div
from fasthtml.xtend import Style
from pyskills.core import resolve
import dialoghelper
from dialoghelper.core import add_html, add_msg, is_usable_tool, read_msg, update_msg, find_msgs, find_dname
from toolslm.funccall import get_schema, resolve_nm
from fastgit import Git

In [ ]:
#| hide
import asyncio, time
from anyio import sleep
from anyio.from_thread import start_blocking_portal
from tempfile import TemporaryDirectory
from IPython.display import Markdown
from fastcore.test import *
from fastcore.basics import first_match
from fastcore.xtras import dict2obj
from dialoghelper.core import del_msgs, msg_idx, run_msg
import dialoghelper.tmux
import dialoghelper.tracetools

## globals - what's available on startup?
Exploring solveit sym injections

::: {.content-hidden}
Run below first after a restart.
:::

In [ ]:
#| hide
print(gk)

('__name__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', 'read_url', 'transient', 'run_cmd', 'maybe_await', 'call_tool', 'sys', 'python', 'allow', 'greet', 'craft_chain', '_i', '_ii', '_iii', '_i1', '_1', '_i2', '__os', '_i3', '_i4', '__dialog_name', '_i5', 'core', 'dh_settings', 'Placements', 'mermaid_url', 'msg_insert_line', 'msg_str_replace', 'msg_strs_replace', 'msg_replace_lines', 'msg_del_lines', 'names_containing', 'find_dname', 'xposta', 'xgeta', 'DialogAPIError', 'call_endp', 'call_endpa', 'curr_dialog', 'msg_idx', 'add_html_a', 'add_html', 'add_scr_a', 'add_scr', 'iife_a', 'iife', 'add_mod', 'add_mod_a', 'pop_data_a', 'pop_data', 'fire_event_a', 'fire_event', 'event_get_a', 'event_get', 'trigger_now', 'event_once', 'event_once_a', 'js_run', 'js_run_a', 'js_eval', 'js_eval_a', 'Channel', 'display_response', 'connfiles', 'realpath', 'list_dialogs', 'read_msg', 'find_msgs', 'view_dlg', 'add_msg', 'read

In [ ]:
#| hide
if 'find_dname' in gk: assert f"/{__dialog_name}" == find_dname()
print(f"{g['__name__']=}\n{g['transient']=}\n{g['__os']=}\n{g['__dialog_name']=}")

g['__name__']='__main__'
g['transient']=<function transient>
g['__os']=<module 'os' from '/usr/local/lib/python3.12/os.py'>
g['__dialog_name']='prj/pote/nbs/00_dutil'


In [ ]:
#|hide
# dir(ipykernel_helper)

In [ ]:
#|hide
# await add_msg(symsrc('ipykernel_helper.core'), msg_type='raw')

In [ ]:
#|hide
sys.meta_path

[<_distutils_hack.DistutilsMetaFinder>,
 _frozen_importlib.BuiltinImporter,
 _frozen_importlib.FrozenImporter,
 _frozen_importlib_external.PathFinder,
 __editable___droute_0_0_1_finder._EditableFinder,
 __editable___htmx_bridge_0_0_1_finder._EditableFinder,
 __editable___nbian_0_0_1_finder._EditableFinder,
 __editable___nbutil_0_0_1_finder._EditableFinder,
 __editable___pote_0_1_1_finder._EditableFinder,
 __editable___raininghard_0_0_1_finder._EditableFinder,
 __editable___solveit_0_0_117_finder._EditableFinder,
 <six._SixMetaPathImporter>]

In [ ]:
[k for k in sys.modules if 'solveit' in k]

['__editable___solveit_0_0_117_finder', 'dialoghelper.solveitskill']

In [ ]:
#|hide
slvt_k = [k for k in sys.modules if '__editable___solveit_' in k][0]
print(dir(slvt := sys.modules[slvt_k]))

['MAPPING', 'ModuleSpec', 'NAMESPACES', 'PATH_PLACEHOLDER', 'Path', 'PathFinder', '_EditableFinder', '_EditableNamespaceFinder', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'chain', 'install', 'module_suffixes', 'spec_from_file_location', 'sys']


In [ ]:
#|hide
# if (msg := await read_msg(1, True))['content'].startswith('!cat'):
#     update_msg(msg['id'], content=f"!cat {slvt.__file__}")
#     run_msg(msg['id'])

In [ ]:
#|hide
' '.join(_.__module__ for _ in sys.meta_path)

'_distutils_hack _frozen_importlib _frozen_importlib _frozen_importlib_external __editable___droute_0_0_1_finder __editable___htmx_bridge_0_0_1_finder __editable___nbian_0_0_1_finder __editable___nbutil_0_0_1_finder __editable___pote_0_1_1_finder __editable___raininghard_0_0_1_finder __editable___solveit_0_0_117_finder six'

In [ ]:
#|hide
# s = Path('/usr/local/lib/python3.12/site-packages/__editable___solveit_0_0_89_finder.py').read_text()
# await add_msg(s, msg_type='raw')

In [ ]:
def solveit_version():
    "Return the version of solveit if it is found"
    s = ' '.join(_.__module__ for _ in sys.meta_path)
    mtch = re.match(r'.*__editable___solveit_(\d+)_(\d+)_(\d+)_finder', s)
    return f"{mtch[1]}.{mtch[2]}.{mtch[3]}" if mtch else ''

In [ ]:
solveit_version()

'0.0.117'

In [ ]:
os.environ.get('IN_SOLVEIT')

'True'

In [ ]:
def in_dialog():
    "Check if the code is running in a solveit dialog"
    if not (os.environ.get('IN_SOLVEIT') and in_ipython() and bool(solveit_version())): return False
    try:
        if find_dname(): return True
    except: pass
    return False

In [ ]:
in_dialog()

True

In [ ]:
def get_caller_globals(): 
    "Return the globals of the caller"
    return inspect.currentframe().f_back.f_globals

In [ ]:
test_var = "I'm in globals"
def caller_func():
    local_var = "I'm local"
    return get_caller_globals()

test_is('test_var' in caller_func(), True)
test_is('local_var' in caller_func(), False)

In [ ]:
def sym2mod(sym):
    "Return the module that defines `sym`"
    return inspect.getmodule(resolve(sym))

In [ ]:
mod = sym2mod(RunPython)
mod.__name__, mod.__file__

('safepyrun.core', '/usr/local/lib/python3.12/site-packages/safepyrun/core.py')

In [ ]:
for sym in gk:
    print(sym, end=' - ')
    print(sym2mod(sym))

__name__ - None


__builtin__ - <module 'builtins' (built-in)>


__builtins__ - <module 'builtins' (built-in)>


_ih - None


_oh - None


_dh - None


... (224 outputs left out)

dsrv - <module 'fasthtml.jupyter' from '/usr/local/lib/python3.12/site-packages/fasthtml/jupyter.py'>


_i8 - None


_i9 - None


g - None


In [ ]:
s2m = {sym:sym2mod(sym) for sym in gk}
set(s2m.values())

{<module 'IPython.core.autocall' from '/usr/local/lib/python3.12/site-packages/IPython/core/autocall.py'>,
 <module '_io' (built-in)>,
 <module 'builtins' (built-in)>,
 <module 'dialoghelper.core' from '/usr/local/lib/python3.12/site-packages/dialoghelper/core.py'>,
 <module 'dialoghelper.exhash' from '/usr/local/lib/python3.12/site-packages/dialoghelper/exhash.py'>,
 <module 'exhash' from '/usr/local/lib/python3.12/site-packages/exhash/__init__.py'>,
 <module 'fastcore.aio' from '/usr/local/lib/python3.12/site-packages/fastcore/aio.py'>,
 <module 'fastcore.basics' from '/usr/local/lib/python3.12/site-packages/fastcore/basics.py'>,
 <module 'fastcore.tools' from '/usr/local/lib/python3.12/site-packages/fastcore/tools.py'>,
 <module 'fasthtml.core' from '/usr/local/lib/python3.12/site-packages/fasthtml/core.py'>,
 <module 'fasthtml.jupyter' from '/usr/local/lib/python3.12/site-packages/fasthtml/jupyter.py'>,
 <module 'ipykernel_helper.core' from '/usr/local/lib/python3.12/site-packages/

In [ ]:
def syms2modinfo(syms):
    "Map module names to `(module name, module file, input symbols)`"
    if isinstance(syms, str): syms = map(str.strip, syms.split(','))
    pairs = [(s if isinstance(s, str) else getattr(s, '__name__', type(s).__name__), inspect.getmodule(resolve(s) if isinstance(s, str) else s)) for s in syms]
    mods = dict.fromkeys(m for _,m in pairs if m)
    return {m.__name__:(m.__name__, getattr(m, '__file__', None), tuple(s for s,x in pairs if x is m)) for m in mods}

In [ ]:
syms2modinfo([RunPython, 'py', 'fd'])

{'safepyrun.core': ('safepyrun.core',
  '/usr/local/lib/python3.12/site-packages/safepyrun/core.py',
  ('RunPython', 'py')),
 'rgapi': ('rgapi',
  '/usr/local/lib/python3.12/site-packages/rgapi/__init__.py',
  ('fd',))}

In [ ]:
syms2modinfo('__CRAFT_error,paste_msgs')

{'dialoghelper.core': ('dialoghelper.core',
  '/usr/local/lib/python3.12/site-packages/dialoghelper/core.py',
  ('paste_msgs',))}

Preserve `None` modules under the `'<dialog>'` key:

In [ ]:
def syms2modinfo(syms):
    "Map module names to `(module name, module file, input symbols)`"
    if isinstance(syms, str): syms = map(str.strip, syms.split(','))
    pairs = [(s if isinstance(s, str) else getattr(s, '__name__', type(s).__name__), inspect.getmodule(resolve(s) if isinstance(s, str) else s)) for s in syms]
    mods = dict.fromkeys(m for _,m in pairs)
    return {(m.__name__ if m else '<dialog>'):(m.__name__ if m else '<dialog>', getattr(m, '__file__', None), tuple(s for s,x in pairs if x is m)) for m in mods}

In [ ]:
mi = syms2modinfo('__CRAFT_error,paste_msgs')
test_eq(mi['<dialog>'], ('<dialog>', None, ('__CRAFT_error',)))
test_eq(mi['dialoghelper.core'][2], ('paste_msgs',))
mi

{'<dialog>': ('<dialog>', None, ('__CRAFT_error',)),
 'dialoghelper.core': ('dialoghelper.core',
  '/usr/local/lib/python3.12/site-packages/dialoghelper/core.py',
  ('paste_msgs',))}

Keep `syms2modinfo` as the public aggregate name and extract `syms2mod` beneath it:

In [ ]:
def syms2mod(syms):
    "Map input symbol names to their defining modules"
    if isinstance(syms, str): syms = map(str.strip, syms.split(','))
    return {s if isinstance(s, str) else getattr(s, '__name__', type(s).__name__):inspect.getmodule(resolve(s) if isinstance(s, str) else s) for s in syms}

def syms2modinfo(syms):
    "Map module names to `(module name, module file, input symbols)`"
    sm = syms2mod(syms)
    mods = dict.fromkeys(sm.values())
    return {(m.__name__ if m else '<dialog>'):(m.__name__ if m else '<dialog>', getattr(m, '__file__', None), tuple(s for s,x in sm.items() if x is m)) for m in mods}

In [ ]:
sm = syms2mod('__CRAFT_error,paste_msgs')
test_is(sm['__CRAFT_error'], None)
test_is(sm['paste_msgs'], dialoghelper.core)

mi = syms2modinfo('__CRAFT_error,paste_msgs')
test_eq(mi['<dialog>'], ('<dialog>', None, ('__CRAFT_error',)))
test_eq(mi['dialoghelper.core'][2], ('paste_msgs',))
mi

{'<dialog>': ('<dialog>', None, ('__CRAFT_error',)),
 'dialoghelper.core': ('dialoghelper.core',
  '/usr/local/lib/python3.12/site-packages/dialoghelper/core.py',
  ('paste_msgs',))}

In [ ]:
def symname(sym): return sym if isinstance(sym, str) else getattr(sym, '__name__', type(sym).__name__)

def syms2mod(syms):
    "Map input symbol names to their defining modules"
    if isinstance(syms, str): syms = map(str.strip, syms.split(','))
    return {symname(s):sym2mod(s) for s in syms}

def syms2modinfo(syms):
    "Map module names to `(module name, module file, input symbols)`"
    sm = syms2mod(syms)
    groups = groupby(sm, key=sm.get)
    return {m.__name__ if m else '<dialog>':(getattr(m, '__file__', None), tuple(ss)) for m,ss in groups.items()}

In [ ]:
sm = syms2mod('__CRAFT_error,paste_msgs')
test_is(sm['__CRAFT_error'], None)
test_is(sm['paste_msgs'], dialoghelper.core)

mi = syms2modinfo('__CRAFT_error,paste_msgs')
test_eq(mi['<dialog>'], (None, ('__CRAFT_error',)))
test_eq(mi['dialoghelper.core'][1], ('paste_msgs',))
mi

{'<dialog>': (None, ('__CRAFT_error',)),
 'dialoghelper.core': ('/usr/local/lib/python3.12/site-packages/dialoghelper/core.py',
  ('paste_msgs',))}

## helpers

### context usage
Measures context usage in a dialog.

In [ ]:
def count_tokens(src, model='cl100k_base'):
    "Count tokens in `src` using `model` encoding"
    return len(tiktoken.get_encoding(model).encode(src))

Use gpt4 tokenizer by default.

In [ ]:
count_tokens("def foo(x:int) -> bool: return x > 0")

13

In [ ]:
#| hide
async def ctxusage(id:str='', dname:str=''):
    msgs = await find_msgs(include_output=False, include_skipped=True, dname=dname)
    id = id or (await read_msg(0)).id
    pos = await msg_idx(id)
    return sum(m.input_tokens + (m.output_tokens or 0) for m in msgs[:pos] if not m.skipped)

In [ ]:
print((await find_msgs(ids='75ad25c7'))[0])

{'id': 'c9331496', 'meta': {}, 'time_run': '2026-09-13T17:36:30.859049+00:00', 'bookmark': None, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'run': None, 'skipped': 0, 'pinned': 0, 'o_collapsed': 0, 'i_collapsed': 0, 'content': 'count_tokens("def foo(x:int) -> bool: return x > 0")', 'output': '13', 'msg_type': 'code', 'exported': False}


In [ ]:
# #| export
# async def ctxusage(fromid:str='', toid:str='', dname:str='', msgs=None):
#     "Count the context usage in the dialog between two messages `fromid`, `toid`"
#     return 0
#     if msgs is None: msgs = await find_msgs(include_output=False, include_skipped=True, include_meta=True, dname=dname)
#     toid = toid or (await read_msg(0, dname=dname)).id
#     start,end = first_match(msgs, lambda x:x.id==fromid) if fromid else 0, first_match(msgs, lambda x:x.id==toid)
#     return sum(m.input_tokens + (m.output_tokens or 0) for m in msgs[start:end] if not m.skipped)

In [ ]:
async def ctxusage(fromid:str='', toid:str='', dname:str='', msgs=None):
    "Estimate tokens in non-skipped dialog messages between `fromid` and `toid`"
    if msgs is None: msgs = await find_msgs(include_output=True, include_skipped=True, dname=dname)
    ids = [m['id'] for m in msgs]
    start = ids.index(fromid) if fromid else 0
    toid = toid or (await read_msg(0, dname=dname))['id']
    end = ids.index(toid)
    src = '\n'.join(f"{m['content']}\n{m.get('output') or ''}" for m in msgs[start:end] if not m['skipped'])
    return int(count_tokens(src)*1.30)

In [ ]:
await ctxusage()

13322

In [ ]:
await ctxusage((await read_msg(-2)).id, (await read_msg(0)).id)

232

In [ ]:
_id = '89d9b4f9'
_s = 'Hi Vic.\n\n```json {.usage}\n{"model": "gpt-5.6-terra", "prompt_tokens": 32547, "completion_tokens": 7, "total_tokens": 32554, "cached_tokens": 3456, "cost": 0.004887300000000001}\n```'
if (await read_msg(1)).id != _id:
    _id = await add_msg(msg_type='prompt', content="Hi, I'm Vic", output=_s)
else: await update_msg(_id, output=_s)

Hi, I'm Vic

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Hi Vic.

In [ ]:
def _strip_tok(s):
    for info,_,start,end in reversed(fenced_blocks(s)):
        if info == 'json {.usage}': s = s[:start].rstrip() + s[end:]
    return s

async def strip_token_usage(m, dname=''):
    "Remove Solveit token usage annotation from the output of a message `m`"
    if m.msg_type=='prompt':
        await update_msg(m.id, dname=dname, output=_strip_tok(m.output))

In [ ]:
m = await read_msg(0, id=_id)
await strip_token_usage(m)
m = await read_msg(0, id=_id)
test_eq(len(fenced_blocks(m.output)), 0)

In [ ]:
async def strip_ctx_usage(dname=''):
    "Remove Solveit token usage annotation from all messages output, and return the old/total token count"
    b = await ctxusage(dname=dname)
    ms = await find_msgs(msg_type='prompt', include_meta=False, trunc_in=True, include_skipped=True, dname=dname)
    for m in ms: await strip_token_usage(m, dname)
    a = await ctxusage(dname=dname)
    return f'{b} → {a} ({a/b:.1%})'

In [ ]:
await strip_ctx_usage()

'13881 → 13881 (100.0%)'

### find/set_var

In [ ]:
# old version from `dialoghelper`; not thread safe
def _find_frame_dict(sentinel:str):
    "Find the globals dict containing sentinel, or calling frame's globals if no sentinel"
    frame = currentframe().f_back.f_back
    if not sentinel: return frame.f_globals
    while frame:
        if sentinel in frame.f_globals: return frame.f_globals
        frame = frame.f_back
    return globals()

def set_var(var:str, val, force:bool=False):
    "Set var to val after finding it in all frames of the call stack"
    frame = _find_frame_dict(var)
    if var not in frame and not force: raise ValueError(f"Could not find {var} in any scope")
    frame[var] = val

def find_var(var:str, default:Any=Parameter.empty):
    "Search for var in all frames of the call stack"
    if var in (frame := _find_frame_dict(var)): return frame[var]
    return default
find_var.notfound = Parameter.empty

In [ ]:
test_is(find_var('aSdF'), inspect._empty)
test_is(find_var('aSdF', None), None)

set_var('aSdF', 13, True)
test_is(aSdF,13)
with ExceptionExpected(): set_var('aSdFgHj', 17)

### message linking

see <a href="/dialog_?name=prj%2Fdutil%2Fexplorer%2Flinked_msg" target="_blank">linked_msg explorer</a>

In [ ]:
_ts, _te = ('<!-- ', '#| '), (' -->', '')
async def get_tag(name:str, val:str='', msg_type:str='note'):
    "Return a linked tag as message comment"
    return f"{_ts[msg_type=='code']}{name}: {val or (await read_msg(0)).id}{_te[msg_type=='code']}"

In [ ]:
_tagid = r'(?:_[a-f0-9]{8}|[a-f0-9]{8})'
_tagpats = {
    'code': (re.compile(rf'\A#\|\W*(\w+): ({_tagid})\W*$', re.MULTILINE),
             re.compile(rf'^#\|\W*(\w+): ({_tagid})\W*\Z', re.MULTILINE)),
    'note': (re.compile(rf'\A<!-- (\w+): ({_tagid}) -->\W*$', re.MULTILINE),
             re.compile(rf'^<!-- (\w+): ({_tagid}) -->\s*\Z', re.MULTILINE))
}
_tagpats['raw'] = _tagpats['prompt'] = _tagpats['note']
_tagpats[None] = _tagpats['note'] + _tagpats['code']

def has_tag(s:str, msg_type:str=None) -> bool:
    "Check if string contains tags created by get_tag at the start or end of the string"
    if msg_type == 'code' and not (s.startswith('#|') or '\n#|' in s): return False
    if msg_type == 'note' and not (s.startswith('<!--') or '\n<!--' in s): return False
    return any(re.search(pat, s) for pat in _tagpats[msg_type])

In [ ]:
def find_tag(s: str, msg_type: str = None) -> str:
    "Find tag in a string and return their details"
    for pat in _tagpats[msg_type]:
        if match := re.search(pat, s): return f"{match.group(1)}: {match.group(2)}"
    return ''

In [ ]:
async def get_linked(id:str) -> str:
    if (msg := (await read_msg(0, id=id))).get('id','') == id:
        if tag := find_tag(msg.content, msg.msg_type):
            k,v = tag.split(': ')
            if k == 'linkedto': return v
    return ''

In [ ]:
if in_dialog() and __name__ != '__main__': get_ipython().xpush(__linked_msgs={})

In [ ]:
@delegates(add_msg)
async def link_msg(
    content:str=None,  # Content of the linked message
    id:str=None,  # ID of the message to link to, or current message if not provided
    pos:str='end',  # Position of the tag in the message
    **kwargs  # Additional keyword arguments for `add_msg` or `update_msg`
) -> str:  # id of linked message
    "Add or update a message linked to `anchor` message. Note only one linked msg per anchor."
    anchor_id, linked = id or (await read_msg(0)).id, find_var('__linked_msgs')
    def _with_tag(tag, c): return f"{tag}\n{c}" if pos=='start' else f"{c}\n{tag}"
    if linked_id := linked.get(anchor_id):
        if (msg := await read_msg(0, id=linked_id)).get('id','') == linked_id:
            tag = await get_tag('linkedto', anchor_id, kwargs.get('msg_type', msg.msg_type))
            if content: kwargs['content'] = _with_tag(tag, content)
            linked[anchor_id] = await update_msg(linked_id, **kwargs)
            return linked[anchor_id]
    tag = await get_tag('linkedto', anchor_id, kwargs.get('msg_type', 'note'))
    linked[anchor_id] = await add_msg(_with_tag(tag, content or '.'), id=anchor_id, **kwargs)  # NOTE: bug note message w/ only comment
    return linked[anchor_id]

In [ ]:
linkedid = await link_msg("I'm linked to my destiny!")

I'm linked to my destiny!
<!-- linkedto: 3ba03e96 -->

In [ ]:
thisidx = await msg_idx()
await run_msg(linkeeid := (await read_msg(-2)).id)
test_eq(thisidx, await msg_idx(linkeeid)+2)

In [ ]:
async def hydrate():
    "Traverse dialog looking for linked messages to update `__linked_msgs`"
    linked = find_var('__linked_msgs')
    linked.clear()
    msgs = await find_msgs(include_meta=True, include_output=False, include_skipped=True)
    ids = {_.id for _ in msgs}
    for msg in msgs:
        if tag := find_tag(msg.get('content', ''), msg.get('msg_type')):
            k, v = tag.split(': ')
            if k == 'linkedto' and v in ids: linked[v] = msg['id']
    return linked

In [ ]:
links = await hydrate()
test_eq(links[linkeeid], linkedid)
display(links)

{'3ba03e96': '0d185766',
 '89419261': '55195652',
 'f40fb050': 'a60aad75',
 '9bd38272': 'f69b52ad',
 '11232531': 'b13f8b37',
 'aad5e6e4': '03a1d05f'}

### wait
> helpers for blocking call w/out freezing the kernel

In [ ]:
async def wait_for(pred, timeout=2, interval=.01):
    "Wait until possibly-async `pred` is truthy, returning its value"
    if timeout < 0: raise ValueError("timeout must be non-negative")
    if interval <= 0: raise ValueError("interval must be positive")
    end = asyncio.get_running_loop().time() + timeout
    while True:
        if res := await maybe_await(pred()): return res
        remaining = end - asyncio.get_running_loop().time()
        if remaining <= 0: raise TimeoutError(f"Condition was not met within {timeout}s")
        await asyncio.sleep(min(interval, remaining))

These cover the promised behavior: polling success, async predicates, and immediate timeout.

In [ ]:
n = 0
def later():
    global n
    n += 1
    return n == 3

test_eq(await wait_for(later, timeout=.1, interval=.001), True)
test_eq(n, 3)

async def async_yes(): return True
test_eq(await wait_for(async_yes), True)

try: await wait_for(lambda: False, timeout=0)
except TimeoutError: pass
else: raise AssertionError("expected TimeoutError")

### next dupe in folder

In [ ]:
def next_dup(fp, marker='_dup'):
    "Get next available duplicate number in fp parent directory"
    fp = Path(fp)
    p, nm, suff = fp.parent, fp.stem.split('.')[0], ''.join(fp.suffixes)
    nm = m[1] if (m := re.match(r'^(.+)'+re.escape(marker)+r'\d+$', nm)) else nm
    # if not p.exists(): i, p/f"{nm}{marker}{i}{suff}"
    dupr = re.compile(r'^'+re.escape(nm+marker)+r'(\d+)$')
    nums = sorted({int(m[1]) for f in p.iterdir() if (m := re.match(dupr, f.stem.split('.')[0]))})
    # if not nums: return 0, p/f"{nm}{suff}"
    for i,n in enumerate(nums, 1):
        if i != n: return i, p/f"{nm}{marker}{i}{suff}"
    return len(nums)+1, p/f"{nm}{marker}{len(nums)+1}{suff}"

A given directory can have files with names that match "*_dup1.*", "*_dup2.*", etc. There can be gaps in the sequence, e.g., 1, 2, 5, 7. `next_dup` get the next available number starting from 1 and considering gaps.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img.jpg', 'img_dup3.jpg'): (p/_).touch()
    n, fp = next_dup(p/'img.jpg')
test_eq((n, fp), (1, p/'img_dup1.jpg'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img.jpg', 'img_dup1.jpg'): (p/_).touch()
    n, fp = next_dup(p/'img.jpg')
test_eq((n, fp), (2, p/'img_dup2.jpg'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('test_dup1.txt', 'test_dup2.txt', 'test_dup5.txt', 'test_dup7.txt'): (p/_).touch()
    n, fp = next_dup(p/'test_dup2.txt')
test_eq((n, fp), (3, p/'test_dup3.txt'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    (p/'_test_dup23.txt').touch()
    n, fp = next_dup(p/'test_dup3.txt')
test_eq((n, fp), (1, p/'test_dup1.txt'))

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp)
    for _ in ('img_dup2.jpg', 'img_dup3.jpg'): (p/_).touch()
    n, fp = next_dup(p/'test_dup3.txt')
test_eq((n, fp), (1, p/'test_dup1.txt'))

In [ ]:
def next_filename(path:str):
    "Get next available duplicate filename for path, e.g., 'img.jpg' -> 'img_dup1.jpg'"
    _,fp = next_dup(Path(path))
    return str(fp)

### simple ids

In [ ]:
def gen_id(): return f"_{uuid.uuid4().hex[:8]}"

In [ ]:
gen_id()

'_0e01492c'

### object traversal

In [ ]:
_empty = inspect.Parameter.empty

In [ ]:
#| hide
def val_at(o, sym: str, default=_empty, sep='.'):
    "Traverse nested `o` looking for attributes/items specified in dot-separated `sym`."
    if not isinstance(sym, str): raise TypeError(f'{sym=!r} is not a string')
    try:
        for a in sym.split(sep):
            if a[0]=='-' or a[0].isdigit(): a = int(a)
            try: o = o[a]
            except Exception:
                if isinstance(a, int):
                    a = str(a)
                    try: o = o[a]
                    except Exception: pass
                o = getattr(o, a)
    except Exception as e:
        if default is not _empty: return default
        raise e
    return o

In [ ]:
#| hide
records = [
    {'id': 1, 'items': [2, 3], 'meta': {'count': 4}}, 
    {'id': 5, 'items': [6, 7], 'nested': [{'val': 81}, {'val': 82}]}, 
    {'id': 9, 'items': [10, 11], 'meta': {'count': 12}}
]

test_eq(val_at(records, '0.id'), 1)
test_eq(val_at(records, '1.items'), [6, 7])
test_eq(val_at(records, '2.meta'), {'count': 12})

::: {.content-hidden}
Works with lists of dicts (common in API responses)
:::

In [ ]:
#| hide
test_fail(lambda: val_at({}, 'a.b'))
test_fail(lambda: val_at([], 'a.b'))
test_fail(lambda: val_at({'a': 1}, 'a.b'))
test_fail(lambda: val_atpath({'a': 1}, 'a', 'b'))

# With default, no error
test_eq(val_at({'a': 1}, 'a.b', None), None)

::: {.content-hidden}
Error handling: raises when path not found (unless default provided)
:::

In [ ]:
#| hide
j2 = {
    "app": {
        "Garden": {
            "Flowers": {
                "Red flower": "Rose",
                "White Flower": "Jasmine",
                "Yellow Flower": "Marigold"
            }
        },
        "Fruits": {
            "Yellow fruit": ["Mango", {"Banana": ["Canary Island", "Puerto Rico"]}],
            "Green fruit": "Guava",
            "White Flower": "groovy"
        },
        "Trees": {
            "label": {
                "Yellow fruit": "Pumpkin",
                "White Flower": "Bogan"
            }
        },
        "Numbers": [1, 2, 3, 4, 5],
        "Boolean": True,
        "Null": None
    }
}

j2_str = j2#json.dumps(j2)

test_eq(val_at(j2_str, 'app.Fruits.Yellow fruit.1.Banana.0'), 'Canary Island')
test_eq(val_at(j2_str, 'app.Garden.Flowers.Red flower'), 'Rose')
test_eq(val_at(j2_str, 'app.Numbers.2'), 3)
test_eq(val_at(j2_str, 'app.Boolean'), True)
test_eq(val_at(j2_str, 'app.Null'), None)
test_fail(lambda: val_at(j2_str, 'app.NonExistent'))
test_fail(lambda: val_at(j2_str, 'app.Fruits.Yellow fruit.3'))
test_is(val_at(j2_str, 'app.Fruits.Yellow fruit.3', None), None)

In [ ]:
#| hide
j2_obj = dict2obj(j2)

test_eq(val_at(j2_obj, 'app.Fruits.Yellow fruit.1.Banana.0'), 'Canary Island')

::: {.content-hidden}
New version of `val_at` that also acepts indexing, i.e. sym can be dot separated with indexing:
- app.Fruits.Yellow fruit[1].Banana[0] <=> app.Fruits.Yellow fruit.1.Banana.0
- app[Fruits][Yellow fruit][1][Banana][0] <=> app.Fruits.Yellow fruit.1.Banana.0
- app[Fruits].Yellow fruit.1.Banana[0] <=> app.Fruits.Yellow fruit.1.Banana.0
- 2.meta <=> [2].meta
- meta.2 <=> meta[2]
:::

In [ ]:
def at_(
    o, # Object to traverse (dict, list, object, or nested combination)
    sym: str, # Path using dots and/or brackets (e.g., 'a.b[0].c' or 'a[b][c]')",
    default: Any=_empty, # Value to return if path not found (raises exception if not provided)
    sep='.' # Separator for path segments
) -> Any: # Value at the specified path
    "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    try:
        for a in filter(None, sym.split(sep)):
            if a.lstrip('-').isdigit(): a = int(a)
            try: o = o[a]
            except Exception:
                if isinstance(a, int):
                    try: o = o[str(a)]; continue
                    except Exception: pass
                o = getattr(o, a)
    except Exception:
        if default is not _empty: return default
        raise
    return o

`at_` provides flexible path-based access to nested data structures:

**Supported types:** `o` can be/contains any combination of `Sequence`, `Mapping` (dicts, lists, tuples, L, etc), objects with \_\_getitem__, and/or dataclasses, objects with attributes accesible by `getattr`.

**Path syntax:**
- Dot notation: `'a.b.c'` accesses `o['a']['b']['c']` or `o.a.b.c`
- Bracket notation: `'a[b][c]'` is equivalent to `'a.b.c'`: `[x]` is a shorthand for `.x`
- Mixed: `'a.b[0].c'` combines both styles
- Numeric indices: `'items.2'` or `'items[2]'` for list/array access
- Empty path: `''` returns the object itself

**Access priority:** Item access (`[]`) is tried before attribute access (`.`)

**Error handling:** Raises exception if path not found, unless `default` is provided

In [ ]:
test_eq(at_({'a': 13}, 'a'), 13)

test_eq(at_({'a': {'b': 13}}, 'a.b'), 13)
test_eq(at_({'a': dict2obj({'b': 13})}, 'a.b'), 13)

test_eq(at_({'a': {'3': 7}}, 'a.3'), 7)

test_fail(lambda: at_(o, 'app.NonExistent'))
test_fail(lambda: at_(o, '[0'))
test_fail(lambda: at_(None, 'a'))
test_eq(at_(None, 'a', None), None)
test_fail(lambda: at_(None, 'a'))

test_eq(at_(o := {'meta': [1,2,3]}, ''), o)
test_eq(at_(o, 'meta[2]'), 3)
test_eq(at_(o, 'meta.2'), 3)

test_eq(at_('xyz', 'a', None), None)
test_eq(at_((s := 'xyz'), 'split'), s.split)

test_eq(at_([{'a':1}], '[0][a]'), 1)

test_eq(at_(records, '[2].meta'), {'count': 12})
test_eq(at_(records, '2[meta]'), {'count': 12})

j2_list = [{'a':1}, {'b':2}]
test_eq(at_(j2_list, '0'), {'a':1})
test_eq(at_(j2_list, '1'), {'b':2})
test_eq(at_(j2_list, '[0][a]'), 1)
test_eq(at_(j2_list, '0.a'), 1)
test_eq(at_(j2_list, '[1].b'), 2)

test_eq(at_(j2, 'app[Numbers][2]'), 3)
test_eq(at_(j2, 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(j2, 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_eq(at_(dict2obj(j2), 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(dict2obj(j2), 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_is(at_(j2, '[bad][path]', 'default'), 'default')
test_is(at_(j2, '[missing]', None), None)
test_is(at_(j2, 'app[missing]', 'default'), 'default')
test_is(at_(j2, 'app[NonExistent]', None), None)

In [ ]:
print(13)

13


In [ ]:
test_eq(at_(await read_msg(), 'output').strip(), '13')

I'm a note.

In [ ]:
test_eq(at_(await read_msg(), 'output'), '')

In [ ]:
if not at_(await read_msg(1), 'content').startswith("I'm a prompt"):
    msgid = await add_msg("I'm a prompt", msg_type='prompt', output='are you? really?')
    # _=add_scr(f"selectMsg($('#{msgid}'), {centered: true})")
display(Markdown('### Test markdown'))
print(_msgid := (await read_msg(0)).id)

### Test markdown

86e4f755


I'm a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

are you? really?

In [ ]:
test_eq(at_(await read_msg(), 'output'), 'are you? really?')

I'm a prompt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

are you? really?

In [ ]:
oo = at_(await read_msg(0, True, _msgid), 'output')
oo

'### Test markdown\n86e4f755\n'

Future improvements, more complex paths:

1. **Negative indexing support** - currently `-1` works but only because Python handles it; we could make it explicit
2. **Wildcard/glob patterns** - e.g. `items.*.name` to get all names from a list of dicts
3. **Optional chaining** - e.g. `a?.b?.c` returns `None` instead of raising if any part is missing
4. **Slice support** - e.g. `items[1:3]` or `items[:2]`
5. **Multiple paths** - return values from multiple paths at once
6. **Callable filter** - e.g. `items[lambda x: x['active']].name`

::: {.content-hidden}
Can we improve `_at` somehow? Give me alternatives. I'm interested in conciseness, strength, flexibility; more complex paths will be handle by another function.
:::

::: {.content-hidden}
The main opportunity is extracting a `_step` helper to flatten the nested try/except. Here are two alternatives:

**Option A** — helper + loop (minimal change, much cleaner):

```python
def _step(o, k):
    try: return o[k]
    except Exception:
        if isinstance(k, int):
            try: return o[str(k)]
            except Exception: pass
        return getattr(o, k)

def at_(o, sym, default=_empty):
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    try:
        for a in filter(None, sym.split('.')): o = _step(o, int(a) if a.lstrip('-').isdigit() else a)
    except Exception:
        if default is not _empty: return default
        raise
    return o
```

**Option B** — functional style with `reduce`:

```python
def at_(o, sym, default=_empty):
    ks = [int(a) if a.lstrip('-').isdigit() else a for a in filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split('.'))]
    try: return reduce(lambda o,k: _step(o,k), ks, o)
    except Exception:
        if default is not _empty: return default
        raise
```
Both drop the `sep` param since complex paths are handled elsewhere. Option A is more readable; Option B is more functional/compact but the list comprehension on one line gets long. I'd lean toward **Option A** — the `_step` helper pays for itself in clarity.
:::

::: {.content-hidden}
**1. Reduce-based (most concise)**
:::

In [ ]:
#| hide
# def at_(o, sym, default=_empty, sep='.'):
#     "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
#     def step(o, a):
#         if a.lstrip('-').isdigit(): a = int(a)
#         try: return o[a]
#         except: return getattr(o, str(a))
#     try: return reduce(step, filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split(sep)), o)
#     except: 
#         if default is not _empty: return default
#         # raise

::: {.content-hidden}
**2. Operator.getitem + getattr combo (cleaner step logic)**
:::

In [ ]:
#| hide
# def at_(o, sym, default=_empty, sep='.'):
#     "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
#     def step(o, a):
#         if a.lstrip('-').isdigit(): a = int(a)
#         for fn in (lambda: o[a], lambda: o[str(a)] if isinstance(a,int) else None, lambda: getattr(o,a)):
#             try: return fn()
#             except: pass
#         raise KeyError(a)
#     try: return reduce(step, filter(None, re.sub(r'\[([^\]]+)\]', r'.\1', sym).split(sep)), o)
#     except:
#         if default is not _empty: return default
#         raise

### add_to_namespace

In [ ]:
async def setup_ns(ns=None, **kwargs):
    "Add `kwargs` to the namespace `ns` or current dialog"
    ns = ns or _find_frame_dict('__dialog_name')
    # lc = list(locals().items())[1:]
    thisid = (await read_msg(0)).id
    for k,v in kwargs.items(): ns[k] = v
    msgid = await link_msg('\n\n'.join(f"{k} = {v}" for k,v in kwargs.items()))
    print(thisid, msgid)

setup_dialog = setup_ns

In [ ]:
await setup_ns(
    ORG='', 
    REPO='', 
    USERNAME='civvic', 
    DESCRIPTION='', 
    COMMIT_MESSAGE='', 
    BRANCH_NAME='',
)

89419261 55195652


ORG = 

REPO = 

USERNAME = civvic

DESCRIPTION = 

COMMIT_MESSAGE = 

BRANCH_NAME = 
<!-- linkedto: 89419261 -->

In [ ]:
test_is('USERNAME' in globals(), True)

### display

In [ ]:
def add_style(css:str):
    "Add CSS to the page."
    _stl = Div(Style(css), hx_swap_oob='beforeend:#dialog-container')
    add_html(_stl)

## info

In [ ]:
#| hide
g = Git('.')

In [ ]:
#| hide
g('status')

'On branch dev-next\nYour branch is ahead of \'origin/dev-next\' by 29 commits.\n  (use "git push" to publish your local commits)\n\nChanges not staged for commit:\n  (use "git add <file>..." to update what will be committed)\n  (use "git restore <file>..." to discard changes in working directory)\n\tmodified:   00_basic.ipynb\n\tmodified:   00_dutil.ipynb\n\tmodified:   02_server.ipynb\n\tmodified:   03_user_ws.ipynb\n\tmodified:   ../pote/server.py\n\tmodified:   ../pote/user_ws.py\n\nno changes added to commit (use "git add" and/or "git commit -a")'

In [ ]:
#| hide
g('status', '-bs')

'## dev-next...origin/dev-next [ahead 29]\n M 00_basic.ipynb\n M 00_dutil.ipynb\n M 02_server.ipynb\n M 03_user_ws.ipynb\n M ../pote/server.py\n M ../pote/user_ws.py'

In [ ]:
def info(dname:str='', json:bool=False):
    "Returns information about the dialog"
    br, chngs, ver, dhv = (), (), solveit_version(), dialoghelper.__version__
    g = Git('.' if not dname else (Path.home()/find_dname(dname).lstrip('/')).parent) 
    if g.exists:
        status = g('status', '-bs').splitlines()
        br,*chngs = status
        br = br.removeprefix('## ').split('...')[0].strip() if br.startswith('## ') else ''
    return ({'Solveit': solveit_version(), 'dialoghelper': dialoghelper.__version__, 'git branch': br, 'git changes': chngs}
        if json else 
        f"Solveit: **v. {ver}**  \ndialoghelper: **v. {dhv}**  " + f"\ngit branch: **{br}**  \ngit changes: {chngs}" if br else '')

In [ ]:
Markdown(info('/prj/EXPER/expers/curator/curator'))

Solveit: **v. 0.0.117**  
dialoghelper: **v. 0.2.41**  
git branch: **main**  
git changes: [' M ../../.gitignore', '?? ../../CRAFT.ipynb', '?? ../../__init__.py', '?? ../../meta-EXPER-v001.ipynb', '?? ../../meta-EXPER.ipynb', '?? ../../meta-EXPER_DEV.ipynb', '?? "../../what is an EXPER.ipynb"']

In [ ]:
Markdown(info())

Solveit: **v. 0.0.117**  
dialoghelper: **v. 0.2.41**  
git branch: **dev-next**  
git changes: [' M 00_basic.ipynb', ' M 00_dutil.ipynb', ' M 02_server.ipynb', ' M 03_user_ws.ipynb', ' M ../pote/server.py', ' M ../pote/user_ws.py']

In [ ]:
async def add_info(msgid:str='', dname:str=''):
    "Add a message with information about the dialog"
    return await link_msg(info(dname), id=msgid)

In [ ]:
await add_info();

Solveit: **v. 0.0.117**  
dialoghelper: **v. 0.2.41**  
git branch: **dev-next**  
git changes: [' M 00_basic.ipynb', ' M 00_dutil.ipynb', ' M 02_server.ipynb', ' M 03_user_ws.ipynb', ' M ../pote/server.py', ' M ../pote/user_ws.py']
<!-- linkedto: f40fb050 -->

## summarize


In [ ]:
def summarize(target, context): pass

## tools

### tool info

In [ ]:
#| hide
# fc_tool_info()

In [ ]:
#| hide
# tool_info()

In [ ]:
#| hide
# inspect_tool_info()

In [ ]:
#| hide
# dialoghelper.tmux.tmux_tool_info()

In [ ]:
def get_tool_names(ns): return [k for k,v in ns.items() if callable(v)]

In [ ]:
#| hide
# print(list(get_tool_names(globals())))

::: {.content-hidden}
see <a href="/dialog_?name=prj%2Fdutil%2Fexplorer%2F00_core_isolated" target="_blank">get_tool_names isolated</a>
:::

In [ ]:
def _get_ns(ns):
    if inspect.ismodule(ns): ns = vars(ns)
    elif isinstance(ns, str): ns = {k: get_ipython().user_ns[k] for _ in re.split(r'\s*,\s*', ns) if (k := _.strip())}
    return ns or get_ipython().user_ns

def get_tool_names(
    ns:Mapping|str=None,  # module,mapping,comma-separated str; None uses IPython user namespace
    exclude:Mapping|list[str]=None,  # module/mapping (recursively scanned) or list of symbol names to exclude
    only_exported:bool=False,  # if ns is a module, only include symbols in __all__
    exclude_private:bool=True  # exclude symbols starting with '_'
) -> dict[str,list[str]]:  # module name -> list of tool names
    "Return dict mapping module names to lists of usable tool names from namespace ns (or IPython user namespace if None)."
    ns = _get_ns(ns)
    if exclude: exclude = set(sum(get_tool_names(exclude).values(), []) if not is_listy(exclude) else exclude)
    res, vis, exports = defaultdict(list), defaultdict(set), set(ns.get('__all__', []))
    for k,v in ns.items():
        if exclude_private and k[0] == '_': continue
        if only_exported and k not in exports: continue
        if exclude and k in exclude: continue
        if not hasattr(__builtins__, k) and callable(v):
            try:
                if is_usable_tool(v): 
                    if inspect.isclass(v) and '__call__' not in v.__dict__: continue
                    if get_schema(v): 
                        mod = getattr(v, '__module__', 'unknown')
                        if v not in vis[mod]: res[mod].append(k); vis[mod].add(v)
            except Exception: pass
    return dict(res)

In [ ]:
get_tool_names()

{'IPython.core.autocall': ['exit'],
 'ipykernel_helper.core': ['read_url', 'info_md'],
 'safepyrun.core': ['python', 'py'],
 'prj.vic': ['greet'],
 'fastcore.tools': ['msg_insert_line',
  'msg_str_replace',
  'msg_strs_replace',
  'msg_replace_lines',
  'msg_del_lines',
  'file_insert_line',
  'file_str_replace',
  'file_strs_replace',
  'file_replace_lines',
  'file_del_lines',
  'file_ast_replace',
  'insert_line',
  'str_replace',
  'strs_replace',
  'replace_lines',
  'del_lines',
  'lnhash_at',
  'view_file',
  'create_file',
  'ast_replace'],
 'dialoghelper.core': ['names_containing',
  'curr_dialog',
  'msg_idx',
  'add_html_a',
  'add_html',
  'iife_a',
  'iife',
  'add_mod',
  'add_mod_a',
  'display_response',
  'connfiles',
  'realpath',
  'list_dialogs',
  'read_msg',
  'find_msgs',
  'view_dlg',
  'add_msg',
  'read_msgid',
  'view_msg',
  'del_msgs',
  'run_and_prompt',
  'update_msg',
  'run_msg',
  'copy_msgs',
  'paste_msgs',
  'toggle_header',
  'toggle_bookmark',
  '

In [ ]:
get_tool_names('add_msg,run_msg')

{'dialoghelper.core': ['add_msg', 'run_msg']}

In [ ]:
get_tool_names(dialoghelper.tmux)

{'fastcore.imports': ['ipython_shell',
  'in_ipython',
  'in_colab',
  'in_jupyter',
  'in_notebook',
  'is_usable_tool'],
 'fastcore.basics': ['num_cpus', 'str2float', 'str2list', 'str2date'],
 'fastcore.xtras': ['walk',
  'globtastic',
  'pglob',
  'loads_multi',
  'parse_env',
  'frontmatter',
  'clean_cli_output',
  'rtoken_hex',
  'modify_exception',
  'stringfmt_names',
  'fenced',
  'fenced_blocks',
  'str_diff',
  'console_help',
  'type2str',
  'is_typeddict'],
 'fastcore.aio': ['reawaitable'],
 'dialoghelper.tmux': ['shell_ret',
  'set_default_history',
  'pane',
  'list_panes',
  'panes',
  'list_windows',
  'windows',
  'list_sessions',
  'sessions']}

In [ ]:
delegates(get_tool_names)
def show_tool_names(*args, **kwargs):
    for mn,syms in get_tool_names(*args, **kwargs).items():
        print(mn)
        print('  ', ', '.join(syms))

In [ ]:
show_tool_names()

IPython.core.autocall


   exit


ipykernel_helper.core


   read_url, info_md


safepyrun.core


   python, py


... (30 outputs left out)

anyio


   sleep


anyio.from_thread


   start_blocking_portal


In [ ]:
show_tool_names(dialoghelper.tracetools)

fastcore.imports


   ipython_shell, in_ipython, in_colab, in_jupyter, in_notebook, is_usable_tool


fastcore.basics


   num_cpus, str2float, str2list, str2date


fastcore.xtras


   walk, globtastic, pglob, loads_multi, parse_env, frontmatter, clean_cli_output, rtoken_hex, modify_exception, stringfmt_names, fenced, fenced_blocks, str_diff, console_help, type2str, is_typeddict


... (2 outputs left out)

toolslm.inspecttools


   resolve


dialoghelper.tracetools


   tracetool


In [ ]:
show_tool_names(dialoghelper.core, only_exported=True)

dialoghelper.core


   names_containing, curr_dialog, msg_idx, add_html_a, add_html, iife_a, iife, add_mod, add_mod_a, display_response, connfiles, realpath, list_dialogs, read_msg, find_msgs, view_dlg, add_msg, read_msgid, view_msg, del_msgs, run_and_prompt, update_msg, run_msg, copy_msgs, paste_msgs, toggle_header, toggle_bookmark, toggle_export, toggle_comment, url2note, create_or_run_dialog, stop_dialog, load_dialog, rm_dialog, run_code_interactive, solveit_docs, dialog_link, spawn_agent, search, searches, web_answer


fastcore.tools


   msg_insert_line, msg_str_replace, msg_strs_replace, msg_replace_lines, msg_del_lines


In [ ]:
show_tool_names(dialoghelper.tmux)

fastcore.imports


   ipython_shell, in_ipython, in_colab, in_jupyter, in_notebook, is_usable_tool


fastcore.basics


   num_cpus, str2float, str2list, str2date


fastcore.xtras


   walk, globtastic, pglob, loads_multi, parse_env, frontmatter, clean_cli_output, rtoken_hex, modify_exception, stringfmt_names, fenced, fenced_blocks, str_diff, console_help, type2str, is_typeddict


fastcore.aio


   reawaitable


dialoghelper.tmux


   shell_ret, set_default_history, pane, list_panes, panes, list_windows, windows, list_sessions, sessions


In [ ]:
def mk_ns_toollist(ns, nms):
    ns = _get_ns(ns)
    return "\n".join(f"- &`{nm}`: {resolve_nm(nm, ns).__doc__}" for nm in nms)

In [ ]:
print(mk_ns_toollist(dialoghelper.tmux, 'shell_ret,pane,list_panes,panes'.split(',')))

- &`shell_ret`: Run shell command locally or over ssh (use host for alias, or ip/user/keyfile)
- &`pane`: Grab the tmux history in plain text
- &`list_panes`: List panes for a session/window (or current if none specified)
- &`panes`: Grab history from all panes in a session/window


In [ ]:
print(mk_ns_toollist(None, ['del_msgs', 'greet']))

- &`del_msgs`: Delete messages from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages.
- &`greet`: Add a note with `wave` message, and code message with setup python path to the current dialog


In [ ]:
@delegates(get_tool_names)
async def add_tools_card(ns:Mapping|str=None, **kwargs):
    "Add a message with all tools in namespace `ns` or caller globals"
    ns = _get_ns(ns)
    mod2tool = get_tool_names(ns, **kwargs)
    content = '<!-- tool card -->\n\n' + '\n\n'.join(f"## {mod}\n\n{mk_ns_toollist(ns, tools)}" for mod,tools in mod2tool.items())
    await link_msg(content)

In [ ]:
#| hide
await add_tools_card()

<!-- tool card -->

## IPython.core.autocall

- &`exit`: An autocallable object which will be added to the user namespace so that
    exit, exit(), quit or quit() are all valid ways to close the shell.

## ipykernel_helper.core

- &`read_url`: Read url from web
- &`info_md`: With `source=False` same as ipython's `?`, otherwise same as `??`

## safepyrun.core

- &`python`: Execute Python with audit-hook safety checks and access to LLM tools, returning last expression.
    `import` works in the usual way. All builtins are available.
    Multiline code blocks can be used, including defining functions and variables.
    **NB**: Locals are exported back to the caller's namespace.
- &`py`: Execute Python with audit-hook safety checks and access to LLM tools, returning last expression.
    `import` works in the usual way. All builtins are available.
    Multiline code blocks can be used, including defining functions and variables.
    **NB**: Locals are exported back to the caller's namespace.

## prj.vic

- &`greet`: Add a note with `wave` message, and code message with setup python path to the current dialog

## fastcore.tools

- &`msg_insert_line`: Insert new_str at specified line number
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted
- &`msg_str_replace`: Replace occurrence(s) of old_str with new_str
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted
- &`msg_strs_replace`: Replace multiple strings simultaneously
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted
- &`msg_replace_lines`: Replace line range with new content; the defaults replace the entire contents
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted
- &`msg_del_lines`: Delete line range; deletion is destructive, so both line numbers must be given explicitly (`1, -1` for all lines)
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add a note showing the deleted content?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted
- &`file_insert_line`: Insert new_str at specified line number
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_str_replace`: Replace occurrence(s) of old_str with new_str
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_strs_replace`: Replace multiple strings simultaneously
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_replace_lines`: Replace line range with new content; the defaults replace the entire contents
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_del_lines`: Delete line range; deletion is destructive, so both line numbers must be given explicitly (`1, -1` for all lines)
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`file_ast_replace`: Apply ast-grep structural pattern replacements to python source
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."

- &`insert_line`: Insert new_str at specified line number
- &`str_replace`: Replace occurrence(s) of old_str with new_str
- &`strs_replace`: Replace multiple strings simultaneously
- &`replace_lines`: Replace line range with new content; the defaults replace the entire contents
- &`del_lines`: Delete line range; deletion is destructive, so both line numbers must be given explicitly (`1, -1` for all lines)
- &`lnhash_at`: `lineno|hash|` exhash address of line `line` of `s`
- &`view_file`: Read file contents, optionally limited to 1-based line range
- &`create_file`: Create a new file with contents. Error if file exists, unless `overwrite`.
- &`ast_replace`: Apply ast-grep structural pattern replacements to python source

## dialoghelper.core

- &`names_containing`: Names containing `s` in IPython user namespace, or caller globals
- &`curr_dialog`: Get the current dialog info.
- &`msg_idx`: Get absolute index of message in dialog.
- &`add_html_a`: Send HTML to the browser to be swapped into the DOM
- &`add_html`: Send HTML to the browser to be swapped into the DOM
- &`iife_a`: Wrap javascript code string in an IIFE and execute it via `add_html`
- &`iife`: Wrap javascript code string in an IIFE and execute it via `add_html`
- &`add_mod`: Wrap javascript code string in a js script module and add it via `add_html`
- &`add_mod_a`: Wrap javascript code string in a js script module and add it via `add_html`
- &`display_response`: Return a special response where `display` is added as markdown/HTML to the prompt output, and `result` is returned to the LLM
- &`connfiles`: Get a running dialogs' connection file path (or dict of all of them if no `dlg_name`.
- &`realpath`: Get the real on-disk path to solveit `subpath`. '/' gets on-disk base path.
- &`list_dialogs`: List dialogs and folders under `subpath`. Folders have `/` suffix.
- &`read_msg`: Get the message indexed in the current dialog.
    NB: Messages in the current dialog above the current message are *already* visible; use this only when you need line numbers for editing operations, or for messages not in the current dialog or below the current message.
    - To get the exact message use `n=0` and `relative=True` together with `id`.
    - To get a relative message use `n` (relative position index).
    - To get the nth message use `n` with `relative=False`, e.g `n=0` first message, `n=-1` last message.
    {dname}
- &`find_msgs`: Often it is more efficient to call `view_dlg` (not `find_msgs`) to see the whole dialog, so you can use it all from then on.
    {dname}
    Message ids are identical to those in LLM chat history, so do NOT call this to view a specific message if it's in the chat history--instead use `view_msg`.
    Do NOT use find_msgs to view message content in the current dialog above the current prompt -- these are *already* provided in LLM context, so just read the content there directly. (NB: LLM context only includes messages *above* the current prompt, whereas `find_msgs` can access *all* messages.)
    To refer to a found message from code, use its `id` field.
- &`view_dlg`: Concise XML view of all messages (optionally filtered by type), not including metadata. Often it is more efficient to call this to see the whole dialog at once (including line numbers if needed), instead of running `find_msgs` or `view_msg` multiple times.
- &`add_msg`: Add/update a message to the queue to show after code execution completes, and optionally run it.
    Code messages are run using python's restricted sandbox.
    **NB**: when creating multiple messages in a row, after the 1st message set `id` to the result of the last `add_msg` call,
    otherwise messages will appear in the dialog in REVERSE order.
    {dname}
- &`read_msgid`: Get message `id`. Message IDs can be view directly in LLM chat history/context, or found in `find_msgs` results.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later.
- &`view_msg`: Views the *content* of message `id`. Same as `read_msgid(...)['content']`, defaulting to `nums=True`.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later.
- &`del_msgs`: Delete messages from the dialog. DO NOT USE THIS unless you have been explicitly instructed to delete messages.
- &`run_and_prompt`: Run `code` and then run `prompt`, returning the resulting message ID.
- &`update_msg`: Update an existing message. Provide either `msg` OR field key/values to update.
    - Use `content` param to update contents.
    - Only include parameters to update--missing ones will be left unchanged.
    {dname}
- &`run_msg`: Adds a message to the run queue. Use read_msg to see the output once it runs.
- &`copy_msgs`: Add `ids` to clipboard.
- &`paste_msgs`: Paste clipboard msg(s) after/before the current selected msg (id).
- &`toggle_header`: Toggle collapsed header state for `id`
- &`toggle_bookmark`: Toggle numbered bookmark (1-9) on a message, clearing it from any other message when setting
- &`toggle_export`: Toggle whether message(s) carry the `#| export` content directive (like the UI export button); all follow the first message's new state
- &`toggle_comment`: Toggle line comments on code message(s). If any lines are uncommented, comments all; otherwise uncomments all.
- &`url2note`: Read URL as markdown, and add note(s) below current message with the result
- &`create_or_run_dialog`: Create a new dialog, or set an existing one running
- &`stop_dialog`: Stop a running dialog kernel
- &`load_dialog`: Run all code messages from `src_dname` into the target dialog's kernel and return dialog contents. Do not call from python; use directly as an LLM tool.
- &`rm_dialog`: Delete a dialog (or folder) and associated records, stopping the kernel if running
- &`run_code_interactive`: Insert code into user's dialog and request for the user to run it. Use other functions where possible, 
    but if they can not find needed information, *ALWAYS* use this instead of guessing or giving up.
    IMPORTANT: This tool is TERMINAL - after calling it, you MUST stop all tool usage 
    and wait for user response. Never call additional tools after this one.
- &`solveit_docs`: Full reference documentation for Solveit - use this to answer questions about how to use Solveit.
    **NB**: The whole docs fit in LLM context, so read the whole thing, don't search/filter it. *Always* re-run rather than relying on truncated history or assumptions.
- &`dialog_link`: Return an IPython HTML link to open a dialog in Solveit.
    After calling this, output the resulting HTML anchor tag exactly as returned—do not wrap in a fenced code block or convert to markdown link format.
- &`spawn_agent`: Spawn a subagent to complete a task defined by `prompt`. Must be run as a tool - not from Python.
    The subagent's context and tools is defined by the parent prompt's history
- &`search`: Get search results for a query and return as text. Rarely used instead of `web_answer`.
- &`searches`: Get search results for multiple queries in parallel. Rarely used instead of `web_answer`.
- &`web_answer`: Use a search agent to search for all of `qs`, choose suitable pages to read, and answer `pr` based on the page contents.

## dialoghelper.exhash

- &`lnhashview_msg`: Show lnhash-addressed lines of a message
- &`msg_exhash`: Verified line-addressed editor. Apply command tuples to msg `id` contents, return lnhash diff.
    **NB**: every command is a tuple whose first element is an address string.
    **NB**: hashes are checked against the current text immediately before each command runs; order commands last->first.

    Use `lnhashview_msg(id)` to get hash-verified addresses before first use.

    Address strings:
        Single:   ``12|a3f2|``
        Range:    ``12|a3f2|,15|b1c3|``
        Last:     ``$`` (last line)
        Whole:    ``%`` (whole file, same as ``1,$``)
        Special:  ``0|0000|`` targets before line 1 (only with a or i)

    Command tuples:
        (addr, "s", pattern, replacement[, flags])
                           Substitute (Rust regex syntax). Replacement supports $1, $0, ${name}.
                           Flags: g=all, i=case-insensitive. args may contain literal / & newlines
        (addr, "d")        Delete line(s)
        (addr, "a", text)  Append payload after line
        (addr, "i", text)  Insert payload before line
        (addr, "c", text)  Change/replace with payload
        (addr, "j")        Join with next line; with range, joins all
        (addr, "m", dest)  Move line(s) after dest address
        (addr, "t", dest)  Copy line(s) after dest address
        (addr, ">", n)     Indent n levels (default 1, 4 spaces each)
        (addr, "<", n)     Dedent n levels (default 1)
        (addr, "sort")     Sort lines alphabetically
        (addr, "p")        Print (include in output without changing)
        (addr, "g", payload), (addr, "g!", payload), (addr, "v", payload)
                           Global commands; payload uses compact ex syntax such as ``/pattern/d``
        (addr, "y", payload) Transliterate; payload uses compact ex syntax such as ``/abc/ABC/``

    `cmds` is a required list of command tuples. Text fields may contain newlines, including
    multiline a/i/c payloads and s pattern/replacement fields; payloads are inserted literally.
    Do NOT add a trailing `.` line — the string boundary ends the block. A trailing `.` will be inserted literally (with a warning).

## exhash

- &`line_hash`: Return a 4-char lowercase hex hash for a single line of text.
- &`lnhash`: Return an lnhash address ``lineno|hash|`` for ``line`` at 1-based ``lineno``.
- &`lnhashview`: Return lines formatted as space-padded ``lineno|hash|content``. Optional 1-based ``start``/``end`` filter the range; ``end`` past EOF is clamped.
- &`lnhashview_file`: Return lines formatted as space-padded ``lineno|hash|content`` for file at ``path``. Optional 1-based ``start``/``end`` filter the range; ``end`` past EOF is clamped.
- &`lnhashview_cell`: Return lines formatted as ``lineno|hash|content`` for the source of notebook cell ``cell_id`` in ipynb file at ``path``. ``cell_id`` may be an exact id or unique prefix; optional 1-based ``start``/``end`` filter the range.
- &`lnhashview_cells`: Return grouped lnhash views for explicit notebook cell ids. Each group starts with ``# cell <id>``; following lines keep normal ``lineno|hash|content`` format.
- &`exhash`: Verified line-addressed editor. Apply commands to `text`, return an EditResult.
    Python commands are tuple specs; raw command strings are rejected. Use
    ``lnhashview(text)`` or ``lnhash(lineno, line)`` to get hash-verified
    address strings. Each command's hashes are checked immediately before it runs.
    A single-line address can match the line's current hash or its call-start hash,
    allowing commands to stack on one line.

    Addresses, command tuples, and payload rules: the ``exhash.skill`` module
    docstring is the full reference. Engine details beyond it:

    - ``s`` fails if the pattern matches nothing in the addressed range
      (substitutes inside ``g`` subcommands stay lenient).
    - ``s`` also fails if the replacement references a capture group the pattern
      does not define (unknown references would otherwise silently substitute the
      empty string); a literal ``$`` is written ``$$``.
    - ``sw`` controls shift width for ``<`` and ``>`` and defaults to 4.
    - In-place edits record each changed line's call-start hash. Structural edits
      drop records at and below their topmost affected line; range addresses never
      use the recorded-hash fallback.
    - Do not use ``.`` terminators: a final ``.`` line is inserted literally,
      and exhash emits a warning.

    Returns an EditResult with attributes (also accessible as dict keys):
      lines     list of output lines
      hashes    lnhash for each output line
      modified  1-based line numbers of modified/added lines
      deleted   1-based line numbers of removed lines (in original)
      origins   for each output line, the 1-based original line number (None if inserted)
      printed   1-based line numbers explicitly addressed by ``p``

    Call ``res.format_diff(context=1)`` for a unified-diff-style summary.
    Non-empty diffs start with ``--- original`` and ``+++ modified`` headers, except a
    ``p``-only result: that renders as a bare ``lnhashview`` of the printed lines, headerless
    and untruncated. Printed lines inside a real diff always show, as context rows.
    NB: ``file_exhash``/``cell_exhash`` with ``inplace=True`` (their default) do not
    return an EditResult: they return the formatted diff string directly (display-truncated via ``truncate_diff``).

    Examples::

      from exhash import exhash, lnhash, lnhashview
      text = "foo\nbar\n"
      addr = lnhash(1, "foo")           # "1|a1b2|"
      res = exhash(text, [(addr, "s", "foo", "baz")])
      print(res["lines"])                # ["baz", "bar"]
      print(res.format_diff())           # unified-diff-style summary
    
- &`file_exhash`: Read files and notebook cells, apply file-aware exhash commands, and return per-target results or a combined diff.

    Command tuples are the ``exhash.skill`` module docstring's; ``path`` is the
    default file context for unqualified addresses. Prefix source address
    strings, and ``m``/``t`` destination strings, with ``path:`` to target
    another file, or ``path.ipynb:cellid:`` to target one notebook cell's
    source (``cellid`` may be an exact id or unique prefix)::

      ("src/a.py:10|aaaa|,20|bbbb|", "m", "src/b.py:$")

    A range must stay within one file or cell. An ``m``/``t`` destination that
    omits the prefix inherits it from the *first address*, never from ``path``:
    a bare destination like ``$`` targets the source's own file, even when
    ``path`` names another. So whenever the source is qualified, qualify the
    destination too. Escape literal colons in filenames as ``\:`` and literal
    backslashes as ``\\``. Missing files are treated as empty only for commands
    valid against an empty buffer (``0|0000|`` with ``a``/``i``, or as an
    ``m``/``t`` destination); cells are never created: a cell target must
    already exist, or the command raises ``KeyError``.

    By default (``inplace=True``) write changed files only after every command
    succeeds and return the combined diff string (display-truncated via
    ``truncate_diff``); if any command fails, write nothing. Lines addressed by ``p``
    are reported too: a ``p``-only call writes nothing and returns those lines as a bare,
    untruncated ``lnhashview``, and printed rows in a target that also changed ride in its
    diff as context. With more than one reported target, each printed-only group is headed
    by ``# file <path>`` or ``# cell <id>``. Pass ``inplace=False`` to preview instead: a
    ``FileSetEditResult`` is returned with ``files``, ``changed``, ``default_path``,
    ``res[path]`` (cell targets under ``'path:cellid'``), and ``res.format_diff(context=1)``.
    
- &`cell_exhash`: Apply exhash commands to the source of notebook cell ``cell_id`` in ipynb file at ``path``.

    Command tuples are the ``exhash.skill`` module docstring's; use
    ``lnhashview_cell(path, cell_id)`` for addresses.
    ``cell_id`` may be an exact id or unique prefix.

    By default (``inplace=True``) write the edited source back when the source actually
    changed (preserving the cell's original str-or-list-of-lines form; the notebook
    re-serializes in Jupyter's JSON layout) and return the diff string (display-truncated via
    ``truncate_diff``); if any command fails, write nothing. A ``p``-only call writes nothing and
    returns the printed lines as a bare, untruncated ``lnhashview``. Pass
    ``inplace=False`` to preview instead: the EditResult is returned without touching the file.
    

## toolslm.xml

- &`json_to_xml`: Convert `d` to XML.
- &`mk_doctype`: Create a `doctype` named tuple
- &`docs_xml`: Create an XML string containing `docs` in Anthropic's recommended format
- &`files2ctx`: Convert files to XML context, handling notebooks
- &`folder2ctx`: Convert folder contents to XML context, handling notebooks; prints instead of returning when run as a CLI
- &`repo2ctx`: Convert GitHub repo to XML context without cloning; prints instead of returning when run as a CLI

## toolslm.inspecttools

- &`importmodule`: Import a module into the caller's global namespace so it's available for `symsrc`, `symval`, `symdir`, etc.
    Use this before inspecting or using symbols from modules not yet imported.
- &`symsrc`: Get the source code for a symbol.

    Examples:

    - `symsrc("Interval")` -> source code of Interval class if it's already imported
    - `symsrc("sympy.sets.sets.Interval")` -> source code of Interval class
    - `symsrc("_last")` -> source of object from previous tool call
    - For dispatchers or registries of callables: `symnth("module.dispatcher.funcs", n) then symsrc("_last")`
- &`symtype`: Get the type of a symbol and set `_last`.

    Examples:

    - `symtype("sympy.sets.sets.Interval")` -> `<class 'type'>`
    - `symtype("doesnotexist")` -> `'SymbolNotFound`
    - `symtype("_last")` -> type of previous result
- &`symval`: List of repr of symbols' values.

    Examples:
    
    - `symval("sympy.sets.sets.Interval")` -> `[<class 'sympy.sets.sets.Interval'>]`
    - `symval("some_dict.keys")` -> `[dict_keys([...])]`
    - `symval("a,notexist")` -> `['foo','SymbolNotFound']`
- &`symtype_val`: List of 2-ple of (type,repr) of symbols' values.

    Examples:
    
    - `symtype_val("a,c,notexist")` -> `[(<class 'str'>,'foo'),(<class 'int'>,1), 'SymbolNotFound']`
- &`symdir`: Get dir() listing of a symbol's attributes and set `_last`. E.g: `symdir("sympy.Interval")` -> `['__add__', '__and__', ...]`
- &`symnth`: Get the nth value from a dict (or any object with .values()). Sets `_last` so you can chain with `symsrc("_last")` etc.

    Examples:
    
    - `symnth("dispatcher.funcs", 12)` -> 13th registered function
    - `symnth("dispatcher.funcs", 0); symsrc("_last")` -> source of first handler
- &`symlen`: Returns the length of the given symbol
- &`symslice`: Returns the contents of the symbol from the given start to the end.
- &`symsearch`: Search contents of symbol, which is assumed to be str for regex, or iterable for non-regex.
    Regex mode returns (match, start, end) tuples; otherwise returns (item, index) tuples
- &`symset`: Set _ai_sym to the given value
- &`symfiles_folder`: Return XML context of files in the folder containing `sym`'s definition
- &`symfiles_package`: Return XML context of all files in `sym`'s top-level package

## pyskills.core

- &`resolve`: Resolve a dotted symbol string to its Python object, with optional [n] indexing
- &`list_pyskills`: Returns {module: description} for all pyskills. To load a module, use `import {module}` then view `doc({module}).
    **NB**: pyskills are *THE* critical way to extend functionality. *ALWAYS* check for pyskills to help with tasks.
    If unsure whether a particular pyskill might help, load it and grabs its docs to see!
- &`xdir`: Filtered names for public symbols of a module or class (or anything with `__dir__`)
- &`doc`: Docstring of modules, classes, functions, instances or any other Python objects.
- &`docfind`: Search `doc()` recursively through `xdir(o)`, looking at submodules, classes, and functions, to depth `n`
- &`pyskills_dir`: Directory for user pyskills
- &`ensure_pyskills_dir`: Create xdg pyskills dir and .pth file if needed

## rgapi

- &`fd`: Find paths with fd-style filters and gitignore support.
- &`fda`: Async `fd`: find paths on Rust threads without blocking the event loop.
- &`ls`: List a directory like `ls`: one level, directories included, ignore rules off, sorted by name.
- &`rg`: Search files and return `SearchResults`, matched paths, or a count; `lnhashs=True` shows exhash-style addresses.
- &`rga`: Async `rg`: search on Rust threads without blocking the event loop.
- &`rg_iter`: Search files lazily, yielding `SearchLine` rows; `lnhashs=True` shows exhash-style addresses.
- &`rga_iter`: Async `rg_iter`: yield `SearchLine` rows as they are found; early exit cancels the search.

## rgapi.nb

- &`nbrg`: Search `.ipynb` cell sources under `root` in parallel, returning matched cells.
- &`nbrg_iter`: Search `.ipynb` cell sources lazily, yielding `NbCell` rows as they are found.
- &`nbrga`: Async `nbrg`: search notebooks on Rust threads without blocking the event loop.
- &`nbrga_iter`: Async `nbrg_iter`: yield `NbCell` rows as they are found; early exit cancels the search.

## safecmd.core

- &`bash`: Run a bash shell command line safely and return the concatencated stdout and stderr.
    Since it is run with `bash`, special chars like `$` and `*` are handled by the shell, so must be quoted if literal.
    `cmd` is parsed and all calls are checked against an allow-list.
    The default allow-list includes most standard unix commands and git subcommands that do not change state or are easily reverted.
    All operators are supported. Output redirects are validated against allowed destinations (default: ./ and /tmp).
    rm_ params are comma-separated strs.

## __main__

- &`hydrate`: Traverse dialog looking for linked messages to update `__linked_msgs`
- &`solveit_version`: Return the version of solveit if it is found
- &`in_dialog`: Check if the code is running in a solveit dialog
- &`get_caller_globals`: Return the globals of the caller
- &`get_tag`: Return a linked tag as message comment
- &`has_tag`: Check if string contains tags created by get_tag at the start or end of the string
- &`find_tag`: Find tag in a string and return their details
- &`link_msg`: Add or update a message linked to `anchor` message. Note only one linked msg per anchor.
- &`next_filename`: Get next available duplicate filename for path, e.g., 'img.jpg' -> 'img_dup1.jpg'
- &`add_style`: Add CSS to the page.
- &`info`: Returns information about the dialog
- &`add_info`: Add a message with information about the dialog
- &`get_tool_names`: Return dict mapping module names to lists of usable tool names from namespace ns (or IPython user namespace if None).
- &`add_tools_card`: Add a message with all tools in namespace `ns` or caller globals

## inspect

- &`currentframe`: Return the frame of the caller or None if this is not possible.

## fastcore.imports

- &`in_ipython`: Check if code is running in some kind of IPython environment
- &`is_usable_tool`: True if the function has a docstring and all parameters have types, meaning that it can be used as an LLM tool.

## fastcore.xtras

- &`fenced_blocks`: Top-level fenced blocks in `text`, fence-nesting-aware: the inverse of `fenced`

## anyio

- &`sleep`: 
    Pause the current task for the specified duration.

    :param delay: the duration, in seconds

    

## anyio.from_thread

- &`start_blocking_portal`: 
    Start a new event loop in a new thread and run a blocking portal in its main task.

    The parameters are the same as for :func:`~anyio.run`.

    :param backend: name of the backend
    :param backend_options: backend options
    :param name: name of the thread
    :return: a context manager that yields a blocking portal

    .. versionchanged:: 3.0
        Usage as a context manager is now required.

    
<!-- linkedto: 9bd38272 -->

In [ ]:
await add_tools_card(dialoghelper.tmux, only_exported=True)

<!-- tool card -->

## dialoghelper.tmux

- &`shell_ret`: Run shell command locally or over ssh (use host for alias, or ip/user/keyfile)
- &`set_default_history`: Set the default number of lines to capture from tmux history
- &`pane`: Grab the tmux history in plain text
- &`list_panes`: List panes for a session/window (or current if none specified)
- &`panes`: Grab history from all panes in a session/window
- &`list_windows`: List all windows in a session
- &`windows`: Grab history from all panes in all windows of a session as `{'win_num:name': {pane_num: content}}`
- &`list_sessions`: List all tmux sessions
- &`sessions`: Grab history from all panes in all windows of all sessions as `{session: {'win_num:name': {pane_num: content}}}`
<!-- linkedto: 11232531 -->

In [ ]:
await add_tools_card('exit, hydrate')

<!-- tool card -->

## IPython.core.autocall

- &`exit`: An autocallable object which will be added to the user namespace so that
    exit, exit(), quit or quit() are all valid ways to close the shell.

## __main__

- &`hydrate`: Traverse dialog looking for linked messages to update `__linked_msgs`
<!-- linkedto: aad5e6e4 -->

# export -

In [ ]:
#|hide
#|eval: false
from pote.dutil import ctxusage
await ctxusage()

26988

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#| hide
# from dutil.gitutil import Git

In [ ]:
#| hide
# g = Git('')
# g.cstatus()

In [ ]:
#| hide
# g.commit('-m', 'renamed to 00_git')

In [ ]:
#| hide
# g.add('07_git.ipynb')

In [ ]:
#| hide
# g.cdiff('../dutil/logger.py')

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()

True